# Session 3: Tool Interface Design and Tool Distribution
**CCAR-F task statements:** 2.1 (tool interfaces and descriptions) and 2.3 (tool distribution and `tool_choice`) · **Domain:** D2 Tool Design and MCP (18%)
**Duration:** 60 minutes · **Extends:** Exercise 1, the Northwind support agent (Session 2 gave it a working loop; the homework added `track_shipment` and `process_refund`)
**Pre-work:** API course, "Tool use with Claude": tool schemas and multiple tools. Docs: *Define tools*, the `tool_choice` section.

By the end of this session you can:
1. Write tool descriptions that differentiate similar tools, and **measure** the difference instead of guessing.
2. Decide when to fix a description, rename a tool, split a generic tool, or narrow it.
3. Scope toolsets per agent role (least privilege) and pick the right `tool_choice` for a step.

| Time | Section | Mode |
|---|---|---|
| 5 min | 1. Setup and recap: your homework tools | run + discuss |
| 7 min | 2. Essential concepts | short explanation |
| 22 min | 3. Descriptions: measure, rewrite, re-measure, overlap, keyword trap | build + test |
| 13 min | 4. Tool distribution and `tool_choice` | build + test |
| 8 min | 5. Exam-style scenarios | reason aloud |
| 5 min | 6. Wrap-up, save, homework | |

> **Instructor note.** Selection tests make real API calls. One full run of the 12 cases at 2 repeats is 24 short Haiku calls (about 20 to 40 seconds with the thread pool). Run the baseline in section 3a *before* the learner joins as well, so you have numbers to fall back on if the network is slow.

## 1. Setup and recap (5 min)

Same project. Today's changes in `northwind/`:

* `common.py` now has **four tools** (the homework state) as `THIN_TOOLS`. If you wrote your own homework definitions, paste them in before running.
* `selection_cases.py`: 12 ambiguous support requests, each with the tool that *should* be picked first.
* `tool_selection.py`: a harness that sends **one** request per case and records which tool Claude picks. No tools are executed, so it is cheap.
* `tool_sprawl.py`: 14 extra tools for the "18 tools vs 4" experiment.
* `test_tool_choice_offline.py`: offline checks for today's loop extension.

In [ ]:
import os, sys, json
sys.path.insert(0, "northwind")
sys.path.insert(0, "northwind/solutions")

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in the environment that launched Jupyter, then restart the kernel."

import anthropic
from common import MODEL, SYSTEM, THIN_TOOLS, client, pick_tools, show
from selection_cases import CASES, SPRAWL_CASES, SHIPPING_IDS, NONE
from tool_selection import first_call, evaluate, print_eval, compare, spread, lint_tools
print("Model:", MODEL)
for t in THIN_TOOLS:
    print(f"  {t['name']:<15} {len(t['description'].split()):>3} words  \"{t['description']}\"")

### Recap: what Claude actually sees

Claude never sees your Python functions. When it picks a tool, it has exactly three things per tool: the **name**, the **description**, and the **input schema**. Look at the four lines above and answer out loud:

1. A customer asks *"Has 12399 gone out yet?"* Which tool would *you* pick from those descriptions alone? Is the other one wrong?
2. In your homework log, did Claude ever pick `lookup_order` for a shipping question, or `track_shipment` for an items question?

<details><summary>Answers</summary>

1. Either is defensible, which is the problem. `lookup_order` "retrieves order details" and secretly also returns carrier and tracking. With thin descriptions the choice is close to a coin toss, and it can flip between runs.
2. Use the learner's log. Any flip is today's opening example: same intent, different tool, different output shape.
</details>

## 2. Essential concepts (7 min)

### Descriptions are the main selection mechanism (2.1)

The official docs call detailed descriptions *"by far the most important factor in tool performance"* and ask for **at least 3 to 4 sentences** per tool. A good description answers:

| Question | Northwind example |
|---|---|
| What does it do and return? | "Returns carrier, tracking number and delivery date..." |
| When should it be used? | "...for any question about where an order is or when it arrives." |
| When should it **not** be used, and what instead? | "It does not return items or prices: use `lookup_order` for those." |
| What do the inputs mean, in what format? | "`order_id`: 5 digits, e.g. '12345'. A leading '#' is accepted." |
| Edge cases | "An order still processing has no tracking number yet; that is a normal result." |

### The interface-fix ladder: cheapest fix first

1. **Rewrite descriptions.** Always first. It is the root cause most of the time.
2. **Rename** when the name itself misleads (`analyze_content` becomes `extract_web_results`).
3. **Split** a generic tool that hides different purposes and outputs (`analyze_document` becomes `extract_data_points`, `summarize_content`, `verify_claim_against_source`).
4. **Narrow** a tool whose *output* duplicates another's, so the overlap disappears at the interface.

Split vs consolidate are not contradictions. The docs also recommend **consolidating** many near-identical verbs on one resource (`create_pr`, `review_pr`, `merge_pr`) into one tool with an `action` parameter. Rule of thumb: *same operation, different verb* → consolidate. *Different purpose, different output* → split.

### The trap: system prompt keywords

A system prompt rule like "whenever a customer mentions an order, use `lookup_order`" is an instruction, and instructions can **override** a good description. When selection is wrong after you fixed the descriptions, read the system prompt next.

**Exam habit:** fix descriptions before adding routing layers, classifiers, or few-shot examples (official sample Q2).

## 3. Descriptions: measure, rewrite, re-measure (22 min)

### 3a. Baseline: how good are the thin descriptions?

**Predict first:** out of 12 cases, how many will the thin descriptions get right? Which case IDs will miss? Write your guess down.

Each case runs twice. "Accuracy" counts runs where the first tool was an acceptable one. "Consistent" counts cases where both runs picked the same thing. A flip is a reliability bug even when one of the two picks was right.

In [ ]:
for c in CASES:
    print(f"{c['id']:<3} want {'/'.join(sorted(c['expect'])):<28} \"{c['request']}\"")

In [ ]:
baseline = evaluate(CASES, THIN_TOOLS, label="thin (homework)", repeats=2)
print_eval(baseline)

**Read the misses before moving on.** For each miss, ask: *which sentence, if it were in the description, would have prevented this?* That sentence goes into your rewrite.

> **Instructor note.** Modern models are good at guessing, so the baseline may already score well on the obvious cases. If it does, focus on (1) the flips, (2) S3 "gone out", R1 (refund before checking the price) and R2/N1 (calling a tool when it should ask or answer), and (3) the keyword and 18-tool experiments below, which reliably show the effect. Do not oversell a small difference: the habit being taught is *measure, change one thing, re-measure*.

### 3b. Exercise: rewrite the four descriptions

Fill in the four `description` strings, and the three missing parameter descriptions on `process_refund`. Use the table in section 2 as your checklist. Rules:

* Say what each tool returns, when to use it, and **when not to, naming the tool to use instead**.
* Give input formats with an example value.
* For `process_refund`, say what must be true *before* it is called. (It moves money. Session 6 will add a hard gate; today the description is the first line of defence.)
* Do not change names or schemas yet. One variable at a time, so the measurement means something.

In [ ]:
import copy
MY_TOOLS = copy.deepcopy(THIN_TOOLS)
mine = {t["name"]: t for t in MY_TOOLS}

mine["lookup_order"]["description"] = (
    "TODO"
)
mine["get_customer"]["description"] = (
    "TODO"
)
mine["track_shipment"]["description"] = (
    "TODO"
)
mine["process_refund"]["description"] = (
    "TODO"
)
# process_refund parameters have no descriptions at all. Fix that too:
props = mine["process_refund"]["input_schema"]["properties"]
props["order_id"]["description"] = "TODO"
props["amount"]["description"] = "TODO"
props["reason"]["description"] = "TODO"

lint_tools(MY_TOOLS)

`lint_tools` is a checklist, not a grader: a long, vague description can pass it. The real test is the selection run below.

Stuck, or short on time? The next cell loads the reference definitions into `MY_TOOLS`. Read them and compare with yours before running the evaluation.

In [ ]:
# Uncomment to use the reference definitions:
# from tools_improved import TOOLS_IMPROVED
# MY_TOOLS = copy.deepcopy(TOOLS_IMPROVED)
# for t in MY_TOOLS: print(f"--- {t['name']}\n{t['description']}\n")

In [ ]:
assert "TODO" not in json.dumps(MY_TOOLS), "Finish the TODOs (or load the reference) first."
improved = evaluate(CASES, MY_TOOLS, label="your descriptions", repeats=2)
print_eval(improved)
print()
compare(baseline, improved)

**Why did you choose this approach?** Answer in one sentence each:
1. A teammate wants to fix the remaining misses with a small classifier model that picks the tool before Claude sees the request. What would you say?
2. Which single sentence in your rewrite do you think did the most work? How could you check?

<details><summary>Answers</summary>

1. That adds a component, latency and a second thing to keep in sync, to fix what is usually a description problem. Fix the descriptions first and only add routing if measured misses remain (sample Q2 reasoning: proportionate first step).
2. Usually the boundary sentence ("use `track_shipment` instead"). To check, delete just that sentence and re-run. That is the same one-variable discipline as above.
</details>

### 3c. The overlap decision: describe it away, or remove it?

`lookup_order` still *returns* carrier and tracking, so it still overlaps `track_shipment`. Two options:

* **Option A, descriptions only** (what you just did). No caller changes.
* **Option B, rename and narrow:** `lookup_order` becomes `get_order_details`, backed by `backend.get_order_details`, which returns no shipping fields. The overlap disappears at the interface.

Measure B. The `aliases` argument scores `get_order_details` as if it were `lookup_order`, so the comparison is fair.

In [ ]:
from tools_improved import TOOLS_RENAMED
renamed = evaluate(CASES, TOOLS_RENAMED, label="B: rename + narrow", repeats=2,
                   aliases={"get_order_details": "lookup_order"})
compare(baseline, improved, renamed)

**Decide and justify:** which option do you ship *today*?

<details><summary>Discussion</summary>

If A already removes the misses, ship A: it fixes the root cause with no changes to callers, tests or other agents (proportionate fix). Choose B when the descriptions cannot separate the tools because the **outputs** genuinely duplicate each other, or when a misleading name keeps attracting the wrong requests. B costs a rename everywhere the tool is referenced. Notice that the Session 2 offline tests use the name `lookup_order`: renaming is never free.
</details>

### Which fix? (2 min)

For each problem pick one: `describe`, `rename`, `split`, `consolidate`, or `narrow`.

In [ ]:
answers = {
    1: "",  # A research agent's `analyze_content` tool is meant only for parsing web search results, but Claude also uses it on PDFs.
    2: "",  # One `analyze_document` tool does extraction, summarisation and claim checking; outputs vary in shape and Claude mixes them up.
    3: "",  # `get_customer` and `lookup_order` have one-line descriptions and Claude confuses them.
    4: "",  # A repo agent has `create_pr`, `update_pr`, `close_pr`, `merge_pr`, `label_pr`, all thin wrappers over one API.
    5: "",  # `lookup_order` returns shipping fields too, and a good description still has not stopped the overlap.
}
KEY = {
    1: ("rename",      "The name invites general use. extract_web_results says exactly what it is for."),
    2: ("split",       "Three purposes with three output shapes: extract_data_points, summarize_content, verify_claim_against_source."),
    3: ("describe",    "Cheapest fix first. Thin descriptions are the root cause (official sample Q2)."),
    4: ("consolidate", "Same resource, different verb: one pull_request tool with an action parameter (docs guidance)."),
    5: ("narrow",      "The overlap is in the output, so remove it at the interface: get_order_details without shipping fields."),
}
for i, (want, why) in KEY.items():
    got = answers[i].strip().lower()
    print(f"{i}. {'ok  ' if got == want else 'MISS'} you: {got or '-':<12} key: {want:<12} {why}")

### 3d. The keyword trap

A rule someone added months ago: *"Whenever a customer mentions an order, always use lookup_order first."* Keep your good descriptions and add only that sentence. **Predict:** what happens to the shipping cases S1 to S4?

In [ ]:
from tools_improved import SYSTEM_KEYWORD, SYSTEM_FIXED
shipping_cases = [c for c in CASES if c["id"] in SHIPPING_IDS]

good_desc   = evaluate(shipping_cases, MY_TOOLS, label="good desc", repeats=2)
keyword     = evaluate(shipping_cases, MY_TOOLS, system=SYSTEM_KEYWORD, label="good desc + keyword rule", repeats=2)
fixed       = evaluate(shipping_cases, MY_TOOLS, system=SYSTEM_FIXED, label="good desc + goal-based prompt", repeats=2)
compare(good_desc, keyword, fixed)
print("\nshipping picks with the keyword rule:", spread(keyword, SHIPPING_IDS))

**Diagnose before fixing:** the descriptions did not change between the first two columns. What did? And why is "add more examples to `track_shipment`'s description" the wrong fix?

<details><summary>Answer</summary>

Only the system prompt changed. An explicit instruction that names a tool by keyword outranks the descriptions, so adding description text fights a stronger signal. Fix the root cause: remove or rewrite the rule so the prompt states goals ("check facts with tools, never guess order numbers") and leaves tool choice to the descriptions. Then re-run the tests. This is a classic exam distractor: a well-written description that "doesn't work" because something upstream overrides it.

If the keyword rule did *not* change the picks on your run, look at the text Claude wrote around the call, and at R1: rules that say "always X first" tend to bite on the cases where X is wrong.
</details>

## 4. Tool distribution and `tool_choice` (13 min)

### 4a. 18 tools vs 4 (2.3)

Support teams add tools over time. `tool_sprawl.py` adds 14 plausible ones, several overlapping yours (`get_order_status`, `list_orders`, `create_return_label`). Same cases, same good descriptions for *your* four tools.

> **Instructor note.** If time is short, run this cell as a demo while the learner reads 4b.

In [ ]:
from tool_sprawl import SPRAWL_TOOLS
four   = evaluate(SPRAWL_CASES, MY_TOOLS, label="4 tools", repeats=2)
eighteen = evaluate(SPRAWL_CASES, MY_TOOLS + SPRAWL_TOOLS, label="18 tools", repeats=2)
compare(four, eighteen)
print("\nshipping questions, distinct tools used:")
print("  4 tools :", spread(four, SHIPPING_IDS))
print("  18 tools:", spread(eighteen, SHIPPING_IDS))

"Accuracy" here means *picked the tool the team designed for that intent*. A pick of `get_order_status` may still produce an answer, but now the same intent goes through two or three different tools with different outputs, permissions and error behaviour. More tools means more decisions per request and more chances of **out-of-role misuse** (a status agent that can also cancel orders).

### Scoped access: give each agent only what its role needs

| Agent role | Tools | Why |
|---|---|---|
| Order-status agent | `get_customer`, `lookup_order`, `track_shipment` | Read-only. It can answer, never change anything. |
| Refunds agent | `get_customer`, `lookup_order`, `process_refund` | Needs the price before refunding; no shipping tools. |
| Any agent, limited cross-role tool | e.g. a narrow `verify_fact` for a synthesis agent | Covers the one cross-role need without handing over the whole toolset. |

**Constrained replacements:** instead of a general tool, expose the narrow one the role needs. `fetch_url` (any URL) becomes `load_document` (approved sources only); a `run_sql` tool becomes `lookup_order`; a `send_email(to, body)` tool becomes `send_order_update(order_id)`.

### 4b. `tool_choice` live

| Setting | Meaning |
|---|---|
| `{"type": "auto"}` | Default. Claude decides whether to call a tool, and which. |
| `{"type": "any"}` | Claude must call *a* tool, its choice of which. |
| `{"type": "tool", "name": "..."}` | Claude must call *this* tool. |
| `{"type": "none"}` | No tool calls. |

Run one request four ways. Look at three things: did it call a tool, which one, and was there any text **before** the `tool_use` block?

> **Model support (check before you build).** Per the current docs, `any` and `tool` return a **400 error on Claude Opus 5.5, Sonnet 5.5, Fable 5.1 and Mythos 5.1**, which recommend `auto` plus strict tool use or structured outputs instead. They also fail with manual extended thinking. Haiku 5.5 (this project's `MODEL`) accepts them. The exam tests the *reasoning* behind each setting, so we learn it here, and the cell below reports a 400 instead of crashing if you switch models.

In [ ]:
request = "Hi, it's alex.rivera@example.com. Can you check on my lantern?"
for choice in [{"type": "auto"}, {"type": "any"}, {"type": "tool", "name": "get_customer"}, {"type": "none"}]:
    print(f"\n=== tool_choice={choice}")
    try:
        names, resp = first_call(request, MY_TOOLS, tool_choice=choice)
        show(resp)
    except anthropic.BadRequestError as e:
        print("  400 from the API:", str(e)[:160])

**Observe and explain:**
1. With `any` and `tool`, was there text before the `tool_use` block? Why not?
2. `tool` forced `get_customer`. Does that guarantee the *second* call is `lookup_order` or `track_shipment`?
3. With `none`, what did Claude do with a question it could not check?

<details><summary>Answers</summary>

1. No. With `any` or `tool` the API prefills the assistant turn to force a tool call, so the model does not write a preamble (docs). Use `auto` with an explicit instruction if you want both.
2. No. Forced selection guarantees the **first** call only. Later steps are decided in follow-up turns, which should go back to `auto` so Claude can choose the next tool and eventually finish with `end_turn`.
3. It answered in text or asked for information. `none` is useful for a final "summarise for the customer" turn where no more lookups should happen.
</details>

### 4c. Exercise: extend `run_agent`

Add three optional parameters to the Session 2 loop below:

1. `tools`: the scoped toolset to send (default: `TOOLS`).
2. `system`: the system prompt (default: `SYSTEM`).
3. `tool_choice`: sent on the **first request only**. Later requests omit it (meaning `auto`).

And one safety rule: if Claude requests a tool that is **not in `tools`**, do not run it. Return an error `tool_result` instead. (The API should not produce this, but least privilege belongs in code, not only in what you send.)

In [ ]:
from common import TOOLS, as_tool_result, run_tool

def run_agent(user_message, client=client, max_iterations=15, verbose=True,
              tools=None, system=None, tool_choice=None):
    # TODO 1: default tools to TOOLS and system to SYSTEM when they are None
    # TODO 2: build the set of allowed tool names from `tools`
    messages = [{"role": "user", "content": user_message}]

    for step in range(1, max_iterations + 1):
        request = dict(model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages)  # TODO 3: use the parameters
        # TODO 4: add tool_choice to the request on the first step only
        response = client.messages.create(**request)
        if verbose:
            print(f"[step {step}] stop_reason={response.stop_reason}")

        if response.stop_reason == "end_turn":
            final_text = "".join(b.text for b in response.content if b.type == "text")
            return final_text, messages + [{"role": "assistant", "content": response.content}]

        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type == "tool_use":
                    output = run_tool(block.name, block.input)   # TODO 5: refuse tools outside `tools`
                    if verbose:
                        print(f"          tool {block.name}({block.input}) -> {str(output)[:80]}")
                    results.append(as_tool_result(block, output))
            messages.append({"role": "user", "content": results})
            continue

        raise RuntimeError(f"Unhandled stop_reason: {response.stop_reason}")

    raise RuntimeError(f"Hit the {max_iterations}-iteration guard rail without end_turn")

### Self-check (offline, no API calls)

Today's six checks, plus the seven Session 2 checks as a regression test: the new parameters must not break the loop.

In [ ]:
from test_loop_offline import check_loop, report
from test_tool_choice_offline import check_s3
print("Session 2 regression"); report(check_loop(run_agent))
print("\nSession 3");          report(check_s3(run_agent))

Stuck after two attempts? Run the next cell to load the reference, read it, run it, and re-run the self-check.

> **Instructor note.** The common mistake is applying `tool_choice` on every step. Check 2 catches it offline; live, it shows up as the guard rail firing, because a forced tool call can never be `end_turn`.

In [ ]:
%load northwind/solutions/agent_loop.py

### 4d. Run it live: scoped agents and a forced first step

Two agents built from the same loop, different toolsets. **Predict** what the read-only status agent does with a refund request.

In [ ]:
STATUS_TOOLS  = pick_tools("get_customer", "lookup_order", "track_shipment", tools=MY_TOOLS)
REFUND_TOOLS  = pick_tools("get_customer", "lookup_order", "process_refund", tools=MY_TOOLS)

q = "Order 12399: the stove arrived broken. Please refund it."
print("=== STATUS agent (read-only)")
final, _ = run_agent(q, tools=STATUS_TOOLS)
print("AGENT:", final)

print("\n=== REFUND agent")
final, history = run_agent(q, tools=REFUND_TOOLS)
print("AGENT:", final)
print("tools used:", [b.name for m in history if m["role"] == "assistant" and isinstance(m["content"], list)
                      for b in m["content"] if getattr(b, "type", "") == "tool_use"])

Now force identity lookup as the first step for a customer who gave an email, and let the loop continue in `auto`.

In [ ]:
final, history = run_agent("alex.rivera@example.com here. Did my lantern arrive?",
                           tools=STATUS_TOOLS, tool_choice={"type": "tool", "name": "get_customer"})
print("AGENT:", final)

**Why did you choose this approach?**
1. Why does the refund agent get `lookup_order` but not `track_shipment`?
2. Forcing `get_customer` first looks like identity verification. Is it a safe way to *guarantee* nobody gets a refund without a verified customer?
3. When would you use `any` rather than forcing a named tool?

<details><summary>Answers</summary>

1. It needs the item price before refunding (least privilege still includes what the role genuinely needs). Shipping data is not part of its job.
2. No. It guarantees only that the first call is `get_customer`; it does not check the result or block a later `process_refund`. A real guarantee needs a programmatic gate on the refund call, which is Session 6 (hooks and prerequisite gates).
3. When some tool must be called but Claude should pick which, e.g. an extraction step with several document-type schemas where a plain-text answer is never acceptable (Session 12).
</details>

## 5. Exam-style scenarios (8 min)

**Start with official sample Q2** (paraphrased here; read the original in the exam guide). Production logs show the support agent often calls `get_customer` when users ask about orders. Both tools have one-line descriptions and accept similar identifiers. Options: few-shot examples, expanded descriptions (input formats, example queries, edge cases, boundaries), a routing layer, or merging the two tools. Talk through each option before looking at the guide's answer (expanded descriptions: root cause, cheapest effective fix).

Then the four questions below. Take two in class and set the rest as homework. Reason aloud: root cause first, then the proportionate fix.

---
**Q1 (2.1).** After you rewrite all tool descriptions, your support agent still sends about 70% of shipping questions to `lookup_order` instead of `track_shipment`. The system prompt contains: "Whenever a customer mentions an order, always use lookup_order first." Which change most directly fixes this?

A. Add three few-shot examples to the system prompt showing `track_shipment` being used for shipping questions.
B. Set `tool_choice` to `track_shipment` whenever the message contains "ship" or "deliver".
C. Remove or rewrite the system prompt rule so it does not tie a keyword to a tool, then re-run the selection tests.
D. Lengthen `track_shipment`'s description with more example queries.

<details><summary>Answer and reasoning</summary>

**C.** The keyword instruction overrides the descriptions; remove the root cause and re-measure.
A and D add more signal against a stronger, conflicting instruction instead of removing it. B is keyword routing in code: brittle, and forcing a tool removes Claude's judgment for every message that happens to mention shipping.
</details>

---
**Q2 (2.1).** A research agent has one `analyze_document(document, mode)` tool used for extracting figures, summarising, and checking whether a source supports a claim. Claude often summarises when asked to verify a claim, and downstream code struggles because the output shape varies by mode. What is the best fix?

A. Split it into `extract_data_points`, `summarize_content` and `verify_claim_against_source`, each describing its purpose and output.
B. Add "choose the correct mode carefully" to the system prompt.
C. Raise `max_tokens` so the tool can return all three outputs every time.
D. Put a classifier model in front of the agent to choose the mode.

<details><summary>Answer and reasoning</summary>

**A.** One generic tool hides three purposes with three output shapes. Purpose-specific tools make the choice explicit and the outputs predictable.
B is prompt guidance for an interface problem. C wastes tokens and makes parsing worse. D adds a routing layer before fixing the interface.
</details>

---
**Q3 (2.3).** In a multi-agent research system, the synthesis subagent was given all 18 tools "for flexibility". It now runs its own web searches mid-synthesis and sometimes ignores the findings it was given. It does occasionally need to confirm a single fact. What should you do?

A. Add "do not search the web" to the synthesis agent's system prompt.
B. Keep all 18 tools but set `tool_choice` to `none` for the synthesis agent.
C. Merge the search and synthesis agents so one agent has all the context.
D. Restrict the synthesis agent to its synthesis tools plus a narrow `verify_fact` tool, leaving broad search to the search subagent.

<details><summary>Answer and reasoning</summary>

**D.** Least privilege by role, plus a limited cross-role tool for the one legitimate need.
A relies on prompt compliance while leaving the capability in place. B also removes the fact check it needs. C undoes the decomposition and makes the context problem worse.
</details>

---
**Q4 (2.3).** In a document pipeline, `extract_metadata` must run before any enrichment tool. With `tool_choice` set to `auto`, Claude sometimes calls an enrichment tool first or replies in plain text. What should you do?

A. Use `tool_choice: {"type": "any"}` on every request.
B. Force `{"type": "tool", "name": "extract_metadata"}` on the first request, and let follow-up turns use `auto` for the later steps.
C. Force `extract_metadata` on every request in the loop.
D. Add "IMPORTANT: ALWAYS call extract_metadata first" to the system prompt.

<details><summary>Answer and reasoning</summary>

**B.** Forced selection guarantees the first call; later steps happen in follow-up turns.
A guarantees *a* tool, not *which* one. C repeats the same call forever and never reaches enrichment or `end_turn`. D is prompt guidance with a non-zero failure rate when a deterministic option exists.
</details>

> **Instructor note.** If the learner raises it: on the newest Opus/Sonnet 5.5 models, `any`/`tool` are not accepted, and the docs point to `auto` plus strict tool use or structured outputs. Answer the exam question on the reasoning it tests (forced first step, `auto` afterwards). In a real build, check the model's support first.

## 6. Wrap-up, save, and homework (5 min)

Save your work so Session 4 starts from it:
* your descriptions go to `northwind/tool_defs.py` (`common.py` loads them automatically from now on, instead of the thin ones);
* the extended loop goes to `northwind/agent_loop.py` (the Session 2 version is kept as `agent_loop_s2.py`).

In [ ]:
import pprint, shutil, pathlib
assert "TODO" not in json.dumps(MY_TOOLS), "Finish your descriptions (or load the reference) before saving."
pathlib.Path("northwind/tool_defs.py").write_text(
    '"""Tool definitions written at the end of Session 3. common.py imports TOOLS from here."""\n\n'
    "TOOLS = " + pprint.pformat(MY_TOOLS, width=100, sort_dicts=False) + "\n", encoding="utf-8")
print("Saved", len(MY_TOOLS), "tool definitions to northwind/tool_defs.py")

loop = pathlib.Path("northwind/agent_loop.py")
backup = pathlib.Path("northwind/agent_loop_s2.py")
if not backup.exists():
    shutil.copy(loop, backup)
shutil.copy("northwind/solutions/agent_loop.py", loop)
print("agent_loop.py now has tools/system/tool_choice (reference). Compare it with your notebook version.")

### Homework (from the curriculum)

* [ ] **Exercise 1:** test the agent with **10 ambiguous requests of your own** (not the 12 from class). Add them to `selection_cases.py` as a new list, log which tool was chosen, improve the descriptions, and re-test. Record before/after accuracy and consistency.
  * Include at least two requests where the right move is to call **no** tool, and one that mixes two concerns.
  * Change one thing at a time between runs.
* [ ] Read the tool use docs section on **`tool_choice`** (Define tools → Forcing tool use), including the model-support table.
* [ ] Re-do exam Q1 and Q4 in a few days without the answers.

### Next session
**Session 4: Structured errors and error propagation (2.2, 5.3).** Today every tool still returns a plain `{"error": "..."}`. Before Session 4, look at what your agent says to the customer when you ask about order `99999`, and when the status agent is asked for a refund. Bring both transcripts.

> **Instructor note.** Before Session 4, skim the learner's 10 requests: any case where Claude called a tool with an invented order ID is a good opening example for "validation error vs valid empty result".